# **종합실습**
* CIFAR-10 데이터 세트
    * AlexNet의 창시자 Alex krizhevsky가 수집하여 만든 데이터 셋
    * 10개 클래스에 32x32 컬러 이미지 60000개로 구성되어 있으며, 클래스당 6000개 이미지가 있습니다. 50000개의 훈련 이미지와 10000개의 테스트 이미지가 있습니다.
* 모델 평가 : accuracy score
    * 지금까지 배운 내용을 총 동원해서 accuracy를 높여 봅시다.

![](https://production-media.paperswithcode.com/datasets/4fdf2b82-2bc3-4f97-ba51-400322b228b1.png)


## 1.환경준비

### (1) 라이브러리 Import

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import *
from sklearn.preprocessing import StandardScaler, MinMaxScaler, LabelEncoder

In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam
from torchvision import datasets
from torchvision.transforms import ToTensor
from torchsummary import summary

### (2) 필요 함수 생성

* 학습을 위한 함수

In [3]:
def train(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)                  # 전체 데이터셋의 크기
    num_batches = len(dataloader)                   # 배치 크기
    tr_loss = 0
    model.train()                                   # 훈련 모드로 설정(드롭아웃 및 배치 정규화와 같은 계층을 훈련 모드로 변경)
    for batch, (X, y) in enumerate(dataloader):     # batch : 현재 배치 번호, (X, y) : 입력 데이터와 레이블
        X, y = X.to(device), y.to(device)           # X.to(device), y.to(device): 입력 데이터와 레이블을 지정된 장치(device, CPU 또는 GPU)로 이동

        # Compute prediction error
        pred = model(X)
        loss = loss_fn(pred, y)
        tr_loss += loss

        # Backpropagation
        loss.backward()             # 역전파를 통해 모델의 각 파라미터에 대한 손실의 기울기를 계산
        optimizer.step()            # 옵티마이저가 계산된 기울기를 사용하여 모델의 파라미터를 업데이트
        optimizer.zero_grad()       # 옵티마이저의 기울기 값 초기화. 기울기가 누적되는 것 방지

    tr_loss /= num_batches          # 모든 배치에서의 loss 평균

    return tr_loss.item()

* 검증을 위한 함수

In [4]:
def evaluate(x_val_tensor, y_val_tensor, model, loss_fn, device):
    model.eval()                        # 모델을 평가 모드로 설정

    with torch.no_grad():               # 평가 과정에서 기울기를 계산하지 않도록 설정(메모리 사용을 줄이고 평가 속도를 높입니다.)
        x, y = x_val_tensor.to(device), y_val_tensor.to(device)
        pred = model(x)
        eval_loss = loss_fn(pred, y).item()    # 예측 값 pred와 실제 값 y 사이의 손실 계산

    return eval_loss, pred

* 학습곡선

In [5]:
def dl_learning_curve(tr_loss_list, val_loss_list, val_acc_list):

    epochs = list(range(1, len(tr_loss_list)+1))
    plt.plot(epochs, tr_loss_list, label='train_err', marker = '.')
    plt.plot(epochs, val_loss_list, label='val_err', marker = '.')
    plt.plot(epochs, val_acc_list, label='val_acc', marker = '.')

    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend()
    plt.grid()
    plt.show()

### (3) device 준비(cpu or gpu)

In [6]:
# cpu 혹은 gpu 사용
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


### (4) 데이터 셋

#### 1) 다운로드

In [7]:
train_dataset = datasets.CIFAR10(root="./", train=True, download=True, transform=ToTensor())
test_dataset = datasets.CIFAR10(root="./", train=False, download=True, transform=ToTensor())

 78%|███████▊  | 132M/170M [29:55<08:36, 73.7kB/s]


KeyboardInterrupt: 

#### 2) 데이터셋 정보

* 데이터셋 일반 정보

In [ ]:
train_dataset, test_dataset

* 데이터셋의 x와 y

* dataset.data : 구조가 이전과 다르다.
    * 파이토치가 받아들이는 구조 : (50000, 3, 32, 32)
    * 제공되는 구조 : (50000, 32, 32, 3)
        * data loader를 이용하면 자동으로 변환해줌 : (n, 3, 32, 32)

In [ ]:
train_dataset.data.shape

* train_dataset.targets : 리스트

In [ ]:
 train_dataset.targets[:10]

* y(target)의 클래스

In [ ]:
classes = train_dataset.classes
classes

* 데이터 1건

In [ ]:
train_dataset[0]

* 데이터 이미지 확인

In [ ]:
n = 200
image, label = train_dataset.data[n], train_dataset.targets[n]

# 이미지 시각화
plt.imshow(image)
plt.title(f"Label: {train_dataset.classes[label]}")
plt.show()

## 2.데이터 준비

* 다운 받은 데이터셋은 다음의 전처리가 완료됨
    * x, y가 분리
    * train, test 분리
    * 스케일링

### (1) train은 데이터로더로 생성

In [ ]:
# 배치 사이즈를 지정해서 Data Loader로 선언하기
batch_size = 64
train_dataLoader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# 첫번째 배치만 로딩해서 살펴보기
for temp_X, temp_y in train_dataLoader:
    print(f"Shape of X [N, C, H, W]: {temp_X.shape}")
    print(f"Shape of y: {temp_y.shape} {temp_y.dtype}")
    break

데이터 로더를 통과한 데이터의 결과를 보면 [64, 3, 32, 32]

### (2) validation, test 준비
* 데이터셋 분할
    * validation : 학습시, 에포크마다 성능 검증용
    * test : 모델 생성 후 최종 검증용
* dataset의 data 속성으로 데이터를 뽑으면 원본데이터가 나옵니다.
    * 스케일링 안된 데이터
    * **5000, 28, 28, 3** : 채널 차원이 뒤에 나옴.
* 그래서 모델링에 사용하려면 두가지 전처리를 다시 해줘야 함
    * 스케일링 : 원본데이터가 0~255 까지 숫자이므로, 255로 나눠주면 됨
    * 차원 위치 변환 : **5000, 3, 28, 28**

In [ ]:
# 1) 데이터와 레이블 추출
test_data = test_dataset.data  # x : (10000, 32, 32, 3)
test_targets = np.array(test_dataset.targets) # y

# 2) 데이터 위치 조정 : [n(0), 세로픽셀(1), 가로픽셀(2), 채널(3)] -> [n(0), 채널(3), 세로픽셀(1), 가로픽셀(2)]
test_data = test_data.transpose((0, 3, 1, 2))  # (10000, 3, 32, 32)

# 3) 텐서로 변환
test_data = torch.tensor(test_data, dtype=torch.float32)
test_targets = torch.tensor(test_targets, dtype=torch.int64)


# 변환된 데이터의 형태 확인
test_data.shape, test_targets.shape


In [ ]:
# 4) 텐서 데이터셋을 validation 5000건, test 5000건으로 나누기
x_val, x_test = test_data[:5000], test_data[5000:]
y_val, y_test = test_targets[:5000], test_targets[5000:]

In [ ]:
# x_val, y_val, x_test, y_test의 shape 확인
x_val.shape, y_val.shape, x_test.shape, y_val.shape

* 스케일링

In [ ]:
x_val = x_val / 255.0
x_test = x_test / 255.0

## 3.모델링1
* Sequential과 Class 선언방식 모두 작성해 봅시다.

### (1) 모델 선언

In [ ]:
n_class = 10

model = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size = 3, stride = 1, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(32, 64, kernel_size = 3, stride = 1, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Flatten(),
    nn.Linear(64 * 8 * 8, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, n_class)
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model.parameters(), lr=0.001)



In [ ]:
class CCNN(nn.Module):
    def __init__(self):
        super(CCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size = 3, stride = 1, padding = 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size = 3, stride = 1, padding = 1)
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 8 * 8, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, n_class)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.relu()
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)

        return x

### (2) 학습

In [ ]:
epochs = 20
tr_loss_list, val_loss_list, val_acc_list = [], [], []

for i in range(epochs):
    tr_loss = train(train_dataLoader, model, loss_fn, optimizer, device)
    val_loss, pred = evaluate(x_val, y_val, model, loss_fn, device)
    val_acc = accuracy_score(y_val, pred.argmax(deim=1))

    tr_loss_list.append(tr_loss)
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)
    print(f"Epoch : {i+1}, Train Loss : {tr_loss:.4f}, Val Loss : {val_loss:.4f}, Val Acc : {val_acc:.4f}")

dl_learning_curve(tr_loss_list, val_loss_list, val_acc_list)

### (3) 모델 평가

In [ ]:
_,pred = evaluate(x_test, y_test, model, loss_fn, device)
pred = nn.functional.softmax(pred, dim=1)
pred = np.argmax(pred.cpu().numpy(), axis=1)

cm = confusion_matrix(y_test  .numpy(), pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(cmap=plt.cm.Blues)
plt.show()

print(classification_report(y_test, pred))

## 4.모델링2
* Sequential과 Class 선언방식 모두 작성해 봅시다.

### (1) 모델 선언

In [ ]:
n_class = 10

model2 = nn.Sequential(
    nn.Conv2d(3, 32, kernel_size = 3, stride = 1, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(32, 64, kernel_size = 3, stride = 1, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),
    nn.Conv2d(64, 128, kernel_size = 3, stride = 1, padding = 1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    nn.Flatten(),
    nn.Linear(128 * 4 * 4, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, n_class)
).to(device)

loss_fn = nn.CrossEntropyLoss()
optimizer = Adam(model2.parameters(), lr=0.001)



In [ ]:
class CCNN(nn.Module):
    def __init__(self):
        super(CCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, kernel_size = 3, stride = 1, padding = 1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size = 3, stride = 1, padding = 1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size = 3, stride = 1, padding = 1)
        self.maxpool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.fc2 = nn.Linear(256, 128)
        self.fc3 = nn.Linear(128, 64)
        self.fc4 = nn.Linear(64, n_class)
        self.relu = nn.ReLU()

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.conv2(x)
        x = self.relu()
        x = self.maxpool(x)
        x = self.conv3(x)
        x = self.relu()
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        x = self.relu(x)
        x = self.fc3(x)

        return x

### (2) 학습

In [ ]:
epochs = 20
tr_loss_list, val_loss_list, val_acc_list = [], [], []

patience = 3
best_loss = float('inf')
counter= 0


for i in range(epochs):
    tr_loss = train(train_dataLoader, model, loss_fn, optimizer, device)
    val_loss, pred = evaluate(x_val, y_val, model, loss_fn, device)
    val_acc = accuracy_score(y_val, pred.argmax(deim=1))

    tr_loss_list.append(tr_loss)
    val_loss_list.append(val_loss)
    val_acc_list.append(val_acc)
    print(f"Epoch : {i+1}, Train Loss : {tr_loss:.4f}, Val Loss : {val_loss:.4f}, Val Acc : {val_acc:.4f}")

    if val_loss < best_loss:
        best_loss = val_loss
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            print(f"Early stopping at epoch {i+1}")
            break

dl_learning_curve(tr_loss_list, val_loss_list, val_acc_list)

### (3) 모델 평가

In [ ]:
_,pred = evaluate(x_test, y_test, model, loss_fn, device)
pred = nn.functional.softmax(pred, dim=1)
pred = np.argmax(pred.cpu().numpy(), axis=1)

cm = confusion_matrix(y_test  .numpy(), pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
disp.plot(cmap=plt.cm.Blues)
plt.show()

print(classification_report(y_test, pred))